---------------------

In [59]:
# 1: Configuration
import requests
import json
import sys
sys.path.insert(0, '.')

# Configuration
BASE_URL = "http://localhost:8000"
TENANT_ID = "3105b788-b5ff-4d56-88a9-532af4ab4ded"
QUERY = "Hướng dẫn tạo bảng giá fcl?"  # Your test query
AGENT_NAME = 'GuidelineAgent'  # or "GuidelineAgent" for direct routing

In [60]:
# Test Chat Endpoint
url = f"{BASE_URL}/api/{TENANT_ID}/chat"

payload = {
    "message": QUERY,
    "session_id": None,
    "user_id": "807318d7-eee4-4f15-93bf-b00f363b006c",
    "metadata": {}
}

if AGENT_NAME:
    payload["agent_name"] = AGENT_NAME

response = requests.post(url, json=payload, headers={"Content-Type": "application/json"})

print(f"Status: {response.status_code}")
data = response.json()

# Pretty print with full depth
print(json.dumps(data, indent=2, ensure_ascii=False, default=str))

# Extract and show tool_calls separately
if response.status_code == 200:
    metadata = data.get('metadata', {})
    tool_calls = metadata.get('tool_calls', [])
    
    print("\n" + "=" * 50)
    print("TOOL CALLS DETAIL")
    print("=" * 50)
    
    for i, call in enumerate(tool_calls):
        print(f"\n--- Tool Call {i+1} ---")
        print(f"Tool: {call.get('tool_name')}")
        print(f"Input: {json.dumps(call.get('tool_args'), ensure_ascii=False)}")
        
        output = call.get('output', {})
        if isinstance(output, dict):
            docs = output.get('documents', [])
            print(f"Documents Retrieved: {len(docs)}")
            
            if docs:
                for j, doc in enumerate(docs):
                    print(f"\n  Document {j+1}:")
                    print(f"    Score: {doc.get('score', 'N/A')}")
                    meta = doc.get('metadata', {})
                    print(f"    Source: {meta.get('source', 'N/A')}")
                    print(f"    Section: {meta.get('section', 'N/A')}")
                    content = doc.get('content', '')
                    print(f"    Content: {content}")  # Full content, no truncation
                    print("-" * 40)
            else:
                print("  No documents retrieved!")
                print(f"  Raw output: {json.dumps(output, ensure_ascii=False, default=str)}")
        else:
            print(f"Output: {output}")

Status: 200
{
  "session_id": "daf4b580-6cf4-4367-bc58-0d528525e4fd",
  "message_id": "7fd120e8-0701-4860-9679-1ac8f614e54f",
  "response": {
    "text": "**Tạo mới bảng giá FCL**\n\nĐể tạo mới bảng giá FCL, người dùng có thể làm theo các bước sau:\n\n**Tạo mới bảng giá FCL**\n\nĐường dẫn: eTMS → Dữ liệu giá → Giá FCL → Bảng giá FCL\n\nQuy trình:\n\nBước 1: Truy cập vào màn hình **Bảng giá FCL (FCL Rate Card List)**.\nBước 2: Nhấn nút **Tạo mới** để mở màn hình **Tạo mới bảng giá FCL**.\nBước 3: Nhập các thông tin cần thiết vào các trường dữ liệu.\nBước 4: Nhấn nút **Lưu** để hoàn tất việc tạo mới bảng giá FCL.\n\nNguồn: 4.5.1. Bảng giá FCL (FCL Rate Card List)"
  },
  "agent": "GuidelineAgent",
  "intent": "create_price_list",
  "format": "text",
  "renderer_hint": {
    "type": "json"
  },
  "metadata": {
    "agent_id": "041d128e-5b98-4f30-a014-8cdee1c0f2fe",
    "tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded",
    "duration_ms": 10472.894430160522,
    "status": "success",
    

______________

## Document Processing Debug Notebook


In [31]:
import sys
sys.path.insert(0, '.')

from src.services.document_processor import get_document_processor
from src.services.embedding_service import get_embedding_service
from pprint import pprint
import json

In [33]:
# Path to your test document
DOCUMENT_PATH = "path/to/your/document.docx"  # Change this
TENANT_ID = "92216a07-7479-41cf-b7cc-772b17d873e1"

# Chunking parameters (same as RAG default)
CHUNK_SIZE = 600
CHUNK_OVERLAP = 150

In [32]:
print("=" * 60)
print("INITIALIZING SERVICES")
print("=" * 60)

doc_processor = get_document_processor()
embedding_service = get_embedding_service()

print(f"Document Processor: {type(doc_processor).__name__}")
print(f"Embedding Service: {type(embedding_service).__name__}")
print()

INITIALIZING SERVICES
Document Processor: DocumentProcessor
Embedding Service: EmbeddingService



In [34]:
# 2: Process Document

print("=" * 60)
print("STEP 1: EXTRACT TEXT FROM DOCUMENT")
print("=" * 60)

try:
    # Extract text from document
    extracted_data = doc_processor.process_document(
        file_path=DOCUMENT_PATH,
        file_type=DOCUMENT_PATH.split('.')[-1]
    )
    
    print(f"Document Type: {extracted_data.get('file_type')}")
    print(f"Total Pages: {extracted_data.get('metadata', {}).get('total_pages', 'N/A')}")
    print(f"Extracted Text Length: {len(extracted_data.get('text', ''))} characters")
    print()
    
    # Show first 500 characters
    print("First 500 characters:")
    print("-" * 40)
    print(extracted_data.get('text', '')[:500])
    print()
    
except Exception as e:
    print(f"Error processing document: {e}")
    import traceback
    traceback.print_exc()


STEP 1: EXTRACT TEXT FROM DOCUMENT
Error processing document: DocumentProcessor.process_document() got an unexpected keyword argument 'file_type'


Traceback (most recent call last):
  File "C:\Users\lucy.le\AppData\Local\Temp\ipykernel_18460\1584552072.py", line 9, in <module>
    extracted_data = doc_processor.process_document(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: DocumentProcessor.process_document() got an unexpected keyword argument 'file_type'


In [35]:
# 3. SPLIT INTO CHUNKS

print("=" * 60)
print("STEP 2: SPLIT TEXT INTO CHUNKS")
print("=" * 60)

try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    
    # Create text splitter (same as RAG service)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    
    # Split text
    full_text = extracted_data.get('text', '')
    chunks = text_splitter.split_text(full_text)
    
    print(f"Total Chunks: {len(chunks)}")
    print(f"Chunk Size: {CHUNK_SIZE} characters")
    print(f"Chunk Overlap: {CHUNK_OVERLAP} characters")
    print()
    
    # Show statistics
    chunk_lengths = [len(chunk) for chunk in chunks]
    print(f"Average Chunk Length: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
    print(f"Min Chunk Length: {min(chunk_lengths)} characters")
    print(f"Max Chunk Length: {max(chunk_lengths)} characters")
    print()
    
except Exception as e:
    print(f"Error splitting text: {e}")
    import traceback
    traceback.print_exc()


STEP 2: SPLIT TEXT INTO CHUNKS
Error splitting text: No module named 'langchain.text_splitter'


Traceback (most recent call last):
  File "C:\Users\lucy.le\AppData\Local\Temp\ipykernel_18460\722052525.py", line 8, in <module>
    from langchain.text_splitter import RecursiveCharacterTextSplitter
ModuleNotFoundError: No module named 'langchain.text_splitter'


In [37]:
# 4. SHOW SAMPLE CHUNKS
print("=" * 60)
print("STEP 3: SAMPLE CHUNKS (First 5)")
print("=" * 60)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Length: {len(chunk)} characters")
    print(f"Content:")
    print(chunk)
    print("-" * 40)

STEP 3: SAMPLE CHUNKS (First 5)


NameError: name 'chunks' is not defined

In [38]:
# 5. GENERATE EMBEDDINGS FOR SAMPLE CHUNKS

print("\n" + "=" * 60)
print("STEP 4: GENERATE EMBEDDINGS (First 3 chunks)")
print("=" * 60)

try:
    for i, chunk in enumerate(chunks[:3]):
        # Generate embedding
        embedding = embedding_service.embed_query(chunk)
        
        print(f"\nChunk {i+1}:")
        print(f"  Text: {chunk[:100]}...")
        print(f"  Embedding Dimension: {len(embedding)}")
        print(f"  First 10 values: {embedding[:10]}")
        
except Exception as e:
    print(f"Error generating embeddings: {e}")
    import traceback
    traceback.print_exc()


STEP 4: GENERATE EMBEDDINGS (First 3 chunks)
Error generating embeddings: name 'chunks' is not defined


Traceback (most recent call last):
  File "C:\Users\lucy.le\AppData\Local\Temp\ipykernel_18460\4145208119.py", line 8, in <module>
    for i, chunk in enumerate(chunks[:3]):
                              ^^^^^^
NameError: name 'chunks' is not defined


In [39]:
# 6. SHOW WHAT WOULD BE STORED IN DATABASE
print("\n" + "=" * 60)
print("STEP 5: DATABASE STORAGE FORMAT (First 3 chunks)")
print("=" * 60)

import uuid
from datetime import datetime

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Document Entry {i+1} ---")
    
    # This is what would be stored in langchain_pg_embedding table
    db_entry = {
        "document": chunk,  # The actual text content
        "cmetadata": {      # Metadata JSONB
            "tenant_id": TENANT_ID,
            "doc_id": str(uuid.uuid4()),
            "source": "document",
            "file_type": DOCUMENT_PATH.split('.')[-1],
            "original_filename": DOCUMENT_PATH.split('/')[-1],
            "chunk_index": i,
            "chunk_total": len(chunks),
            "ingested_at": datetime.utcnow().isoformat(),
            "document_name": "test_document",
        },
        "embedding": "vector(384)",  # Would be actual embedding vector
    }
    
    print(json.dumps(db_entry, indent=2, ensure_ascii=False, default=str))
    print()


STEP 5: DATABASE STORAGE FORMAT (First 3 chunks)


NameError: name 'chunks' is not defined

In [40]:
# 7. SIMULATE RAG QUERY
print("=" * 60)
print("STEP 6: SIMULATE RAG QUERY")
print("=" * 60)

query = "Hướng dẫn tạo bảng giá FCL"
print(f"Query: {query}")
print()

# Generate query embedding
query_embedding = embedding_service.embed_query(query)
print(f"Query Embedding Dimension: {len(query_embedding)}")
print()

# Calculate similarity with first 3 chunks (cosine similarity)
from numpy import dot
from numpy.linalg import norm

print("Similarity Scores (with first 3 chunks):")
for i, chunk in enumerate(chunks[:3]):
    chunk_embedding = embedding_service.embed_query(chunk)
    
    # Cosine similarity
    similarity = dot(query_embedding, chunk_embedding) / (norm(query_embedding) * norm(chunk_embedding))
    
    print(f"\nChunk {i+1}:")
    print(f"  Similarity Score: {similarity:.4f}")
    print(f"  Content Preview: {chunk[:150]}...")

print("\n" + "=" * 60)
print("DOCUMENT PROCESSING COMPLETE")
print("=" * 60)

STEP 6: SIMULATE RAG QUERY
Query: Hướng dẫn tạo bảng giá FCL



Batches: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

Query Embedding Dimension: 384

Similarity Scores (with first 3 chunks):


NameError: name 'chunks' is not defined

In [45]:
# Encode/Decode API Key using Fernet
from cryptography.fernet import Fernet

# Your FERNET_KEY from .env
FERNET_KEY = "kN8j3xP5mR7qT9wV2yB4nL6oC1eH3fA8gD0iK5sU9jM="

def encode_api_key(api_key: str) -> str:
    """Encode API key for database storage."""
    cipher = Fernet(FERNET_KEY.encode())
    encrypted = cipher.encrypt(api_key.encode())
    return encrypted.decode()

def decode_api_key(encrypted_key: str) -> str:
    """Decode API key from database."""
    cipher = Fernet(FERNET_KEY.encode())
    decrypted = cipher.decrypt(encrypted_key.encode())
    return decrypted.decode()

In [46]:
api_key = "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"
encrypted = encode_api_key(api_key)
print(f"Original:  {api_key}")
print(f"Encrypted: {encrypted}")
print()

Original:  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
Encrypted: gAAAAABpHsufNQzvJ_uyQfOp0d42u2D6MauY_-6FsJnENuVNydDmM14kjUPMOcQ1YKZvptOTIOzWpIOIcScKy1fnOjuEgXyW_nV8EwpU4HzL7QpQdIrs3Jr9paAzxGCZs5eeUmAsNUsw5gxC3Qi16h3n3RTciJlf2fg9zHfbqPm58S0IgUIFYc8=



In [50]:
# Check available LLM models
import sys
sys.path.insert(0, '.')

from src.config import SessionLocal
from src.models.llm_model import LLMModel

db = SessionLocal()

print("Available LLM Models:")
print("=" * 60)

models = db.query(LLMModel).all()

for model in models:
    print(f"\nProvider: {model.provider}")
    print(f"Model Name: {model.model_name}")
    print(f"Model ID: {model.llm_model_id}")
    print(f"Active: {model.is_active}")
    print("-" * 40)

db.close()

Available LLM Models:
2025-11-20 15:22:06,836 INFO sqlalchemy.engine.Engine select pg_catalog.version()


select pg_catalog.version()


2025-11-20 15:22:06,846 INFO sqlalchemy.engine.Engine [raw sql] {}


[raw sql] {}


2025-11-20 15:22:06,871 INFO sqlalchemy.engine.Engine select current_schema()


...

SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models


2025-11-20 15:22:06,971 INFO sqlalchemy.engine.Engine [generated in 0.00485s] {}


[generated in 0.00485s] {}



Provider: openrouter
Model Name: openai/gpt-4o-mini
Model ID: b2c3d4e5-f6a7-4859-a5a7-b2c3d4e5f6a7
Active: True
----------------------------------------

Provider: openrouter
Model Name: google/gemini-2.0-flash-exp:free
Model ID: 15f4efa9-46f8-40d2-9cfd-e871977a5634
Active: True
----------------------------------------

Provider: gemini
Model Name: gemini-2.5-flash
Model ID: a1b2c3d4-e5f6-4748-9394-a1b2c3d4e5f6
Active: True
----------------------------------------

Provider: gemini
Model Name: gemini-2.0-flash-lite
Model ID: c5565f19-2d19-4384-82c2-505a7ceac625
Active: True
----------------------------------------
2025-11-20 15:22:07,009 INFO sqlalchemy.engine.Engine ROLLBACK


ROLLBACK


In [51]:
# Update tenant to use OpenRouter
from src.config import SessionLocal
from src.models.llm_model import LLMModel
from src.models.tenant_llm_config import TenantLLMConfig

db = SessionLocal()

# Find OpenRouter model
openrouter_model = db.query(LLMModel).filter(
    LLMModel.provider == "openrouter",
    LLMModel.model_name == "google/gemini-2.0-flash-exp:free"
).first()

if openrouter_model:
    # Update tenant config
    tenant_config = db.query(TenantLLMConfig).filter(
        TenantLLMConfig.tenant_id == "92216a07-7479-41cf-b7cc-772b17d873e1"
    ).first()
    
    if tenant_config:
        tenant_config.llm_model_id = openrouter_model.llm_model_id
        db.commit()
        print("✓ Tenant updated to use OpenRouter!")
    else:
        print("✗ Tenant config not found")
else:
    print("✗ OpenRouter model not found in database")
    print("\nAvailable OpenRouter models:")
    openrouter_models = db.query(LLMModel).filter(LLMModel.provider == "openrouter").all()
    for m in openrouter_models:
        print(f"  - {m.model_name}")

db.close()

2025-11-20 15:22:35,924 INFO sqlalchemy.engine.Engine BEGIN (implicit)


BEGIN (implicit)


2025-11-20 15:22:35,947 INFO sqlalchemy.engine.Engine SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models 
WHERE llm_models.provider = %(provider_1)s AND llm_models.model_name = %(model_name_1)s 
 LIMIT %(param_1)s


SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models 
WHERE llm_models.provider = %(provider_1)s AND llm_models.model_name = %(model_name_1)s 
 LIMIT %(param_1)s


2025-11-20 15:22:35,955 INFO sqlalchemy.engine.Engine [generated in 0.00808s] {'provider_1': 'openrouter', 'model_name_1': 'google/gemini-2.0-flash-exp:free', 'param_1': 1}


...

2025-11-20 15:22:35,985 INFO sqlalchemy.engine.Engine [generated in 0.00397s] {'tenant_id_1': '92216a07-7479-41cf-b7cc-772b17d873e1', 'param_1': 1}


[generated in 0.00397s] {'tenant_id_1': '92216a07-7479-41cf-b7cc-772b17d873e1', 'param_1': 1}


2025-11-20 15:22:36,029 INFO sqlalchemy.engine.Engine COMMIT


COMMIT


✓ Tenant updated to use OpenRouter!


In [55]:
# Check actual database values
import sys
sys.path.insert(0, '.')

from src.config import SessionLocal
from src.models.llm_model import LLMModel
from src.models.tenant_llm_config import TenantLLMConfig

db = SessionLocal()

TENANT_ID = "92216a07-7479-41cf-b7cc-772b17d873e1"

# Get tenant's LLM config
tenant_config = db.query(TenantLLMConfig).filter(
    TenantLLMConfig.tenant_id == TENANT_ID
).first()

if tenant_config:
    print("Tenant LLM Config:")
    print("=" * 50)
    print(f"Config ID: {tenant_config.config_id}")
    print(f"Tenant ID: {tenant_config.tenant_id}")
    print(f"LLM Model ID: {tenant_config.llm_model_id}")
    print()
    
    # Get the actual LLM model details
    llm_model = db.query(LLMModel).filter(
        LLMModel.llm_model_id == tenant_config.llm_model_id
    ).first()
    
    if llm_model:
        print("LLM Model Details:")
        print("=" * 50)
        print(f"Model ID: {llm_model.llm_model_id}")
        print(f"Provider: {llm_model.provider}")  # ← Check this
        print(f"Model Name: {llm_model.model_name}")
        print(f"Display Name: {llm_model.display_name}")
        print(f"Is Active: {llm_model.is_active}")
        print()
        
        # Check if it matches what you expect
        if llm_model.provider == "openrouter":
            print("✓ Provider is correctly set to 'openrouter'")
        else:
            print(f"✗ Provider is '{llm_model.provider}', NOT 'openrouter'!")
            print("\nThis explains why it's using Google API directly.")
    else:
        print("LLM Model not found!")
else:
    print("Tenant config not found!")

db.close()

2025-11-20 15:58:14,821 INFO sqlalchemy.engine.Engine BEGIN (implicit)


BEGIN (implicit)


2025-11-20 15:58:14,871 INFO sqlalchemy.engine.Engine SELECT tenant_llm_configs.config_id AS tenant_llm_configs_config_id, tenant_llm_configs.tenant_id AS tenant_llm_configs_tenant_id, tenant_llm_configs.llm_model_id AS tenant_llm_configs_llm_model_id, tenant_llm_configs.encrypted_api_key AS tenant_llm_configs_encrypted_api_key, tenant_llm_configs.rate_limit_rpm AS tenant_llm_configs_rate_limit_rpm, tenant_llm_configs.rate_limit_tpm AS tenant_llm_configs_rate_limit_tpm, tenant_llm_configs.created_at AS tenant_llm_configs_created_at, tenant_llm_configs.updated_at AS tenant_llm_configs_updated_at 
FROM tenant_llm_configs 
WHERE tenant_llm_configs.tenant_id = %(tenant_id_1)s::UUID 
 LIMIT %(param_1)s


SELECT tenant_llm_configs.config_id AS tenant_llm_configs_config_id, tenant_llm_configs.tenant_id AS tenant_llm_configs_tenant_id, tenant_llm_configs.llm_model_id AS tenant_llm_configs_llm_model_id, tenant_llm_configs.encrypted_api_key AS tenant_llm_configs_encrypted_api_key, tenant_llm_configs.rate_limit_rpm AS tenant_llm_configs_rate_limit_rpm, tenant_llm_configs.rate_limit_tpm AS tenant_llm_configs_rate_limit_tpm, tenant_llm_configs.created_at AS tenant_llm_configs_created_at, tenant_llm_configs.updated_at AS tenant_llm_configs_updated_at 
FROM tenant_llm_configs 
WHERE tenant_llm_configs.tenant_id = %(tenant_id_1)s::UUID 
 LIMIT %(param_1)s


2025-11-20 15:58:14,876 INFO sqlalchemy.engine.Engine [cached since 2139s ago] {'tenant_id_1': '92216a07-7479-41cf-b7cc-772b17d873e1', 'param_1': 1}


...

SELECT llm_models.llm_model_id AS llm_models_llm_model_id, llm_models.provider AS llm_models_provider, llm_models.model_name AS llm_models_model_name, llm_models.context_window AS llm_models_context_window, llm_models.cost_per_1k_input_tokens AS llm_models_cost_per_1k_input_tokens, llm_models.cost_per_1k_output_tokens AS llm_models_cost_per_1k_output_tokens, llm_models.is_active AS llm_models_is_active, llm_models.capabilities AS llm_models_capabilities, llm_models.created_at AS llm_models_created_at 
FROM llm_models 
WHERE llm_models.llm_model_id = %(llm_model_id_1)s::UUID 
 LIMIT %(param_1)s


2025-11-20 15:58:15,062 INFO sqlalchemy.engine.Engine [generated in 0.01094s] {'llm_model_id_1': UUID('15f4efa9-46f8-40d2-9cfd-e871977a5634'), 'param_1': 1}


[generated in 0.01094s] {'llm_model_id_1': UUID('15f4efa9-46f8-40d2-9cfd-e871977a5634'), 'param_1': 1}


LLM Model Details:
Model ID: 15f4efa9-46f8-40d2-9cfd-e871977a5634
Provider: openrouter
Model Name: google/gemini-2.0-flash-exp:free


AttributeError: 'LLMModel' object has no attribute 'display_name'